
# Análisis de sentimientos

## 1. Objetivo

Implementar un proceso de clasificación automática de sentimientos sobre noticias digitales recopiladas mediante fuentes RSS.

El proceso utiliza un modelo Transformer multilingüe preentrenado para identificar el sentimiento general expresado en el título y la descripción de cada noticia.

Se consideran tres categorías:

- **Positivo:** predominio de valoraciones favorables.
- **Negativo:** predominio de valoraciones desfavorables.
- **Neutral:** contenido predominantemente informativo o descriptivo.

## 2. Modelo seleccionado

Se utiliza el modelo `cardiffnlp/twitter-xlm-roberta-base-sentiment`, basado en XLM-RoBERTa y disponible en Hugging Face.

El modelo permite realizar inferencias localmente, sin utilizar servicios externos de clasificación.

Para cada noticia se obtienen:

- Etiqueta de sentimiento predominante.
- Puntuación de la clase positiva.
- Puntuación de la clase negativa.
- Puntuación de la clase neutral.

Las puntuaciones se interpretan como salidas normalizadas del clasificador, no como probabilidades calibradas ni como medidas directas de intensidad emocional.

La evaluación cuantitativa del modelo se documenta por separado y se completará posteriormente mediante etiquetas de referencia revisadas manualmente.


In [1]:
import pandas as pd
import numpy as np
import psycopg
import os
import logging
from transformers import pipeline
from dotenv import load_dotenv

### Configuración de la conexión experimental

Se establece la conexión con la base de datos del caso
experimental, garantizando el aislamiento respecto del
entorno original del asistente.

In [2]:
# Configuración de la base de datos experimental.
EXPECTED_DATABASE = "ida_caso_02"
ENV_PATH = "../.env.ida_caso_02"

# Cargar las variables del entorno experimental.
if not load_dotenv(ENV_PATH, override=True):
    raise RuntimeError("No se pudo cargar el entorno experimental.")

In [3]:
def get_database_connection():
    """Crea una conexión con la base de datos PostgreSQL configurada.

    Returns:
        psycopg.Connection: Conexión activa a la base de datos PostgreSQL.
    """
    return psycopg.connect(
        host=os.getenv("DB_HOST"),
        port=os.getenv("DB_PORT"),
        dbname=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
    )

In [4]:
# Verificar la conexión con la base experimental.
with get_database_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute("SELECT current_database();")
        database = cursor.fetchone()[0]

assert database == EXPECTED_DATABASE, (
    f"Base de datos incorrecta: {database}"
)

print(f"Base experimental activa: {database}")

Base experimental activa: ida_caso_02


In [5]:
# Configurar el registro de eventos.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

logger = logging.getLogger(__name__)

In [6]:

SENTIMENT_CONFIG = {
    "model_name": "cardiffnlp/twitter-xlm-roberta-base-sentiment",
    "max_length": 512,
    "batch_size": 8,
    "device": -1,
}

SENTIMENT_LABELS = [
    "negative",
    "neutral",
    "positive",
]


In [7]:

def load_sentiment_model(config):
    """
    Inicializa el modelo de clasificación de sentimientos.

    Parameters
    ----------
    config : dict
        Parámetros de configuración del modelo.

    Returns
    -------
    transformers.Pipeline
        Pipeline de clasificación de sentimientos.
    """
    logger.info(
        "Cargando modelo de sentimientos: %s",
        config["model_name"],
    )

    classifier = pipeline(
        task="text-classification",
        model=config["model_name"],
        tokenizer=config["model_name"],
        device=config["device"],
    )

    logger.info("Modelo cargado correctamente.")

    return classifier


sentiment_classifier = load_sentiment_model(
    SENTIMENT_CONFIG
)


2026-11-26 20:37:12,279 - INFO - Cargando modelo de sentimientos: cardiffnlp/twitter-xlm-roberta-base-sentiment
2026-11-26 20:37:12,689 - INFO - HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter-xlm-roberta-base-sentiment/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-11-26 20:37:12,698 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-11-26 20:37:12,720 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cardiffnlp/twitter-xlm-roberta-base-sentiment/f2f1202b1bdeb07342385c3f807f9c07cd8f5cf8/config.json?%2Fcardiffnlp%2Ftwitter-xlm-roberta-base-sentiment%2Fresolve%2Fmain%2Fconfig.json=&etag=%22312355cb9d60690ace3ad8fe687a35626399e043%22 "HTTP/1.1 200 OK"
2026-11-26 20:37:12,926 - INFO - HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter-xlm-roberta-base-sentiment/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-11-26 20:37:13,284 - INFO - HTTP Request: GET https://huggingface.co/api/models/cardiffnlp/twitter-xlm-roberta-base-sentiment/commits/main "HTTP/1.1 200 OK"
2026-11-26 20:37:13,355 - INFO - HTTP Request: GET https://huggingface.co/api/models/cardiffnlp/twitter-xlm-roberta-base-sentiment/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-11-26 20:37:13,459 - INFO - HTTP Request: GET https://huggingface.co/api/models/cardiffnlp/twitter-xlm-roberta-base-sentiment/discussions?p=0 "HTTP/1.1 200 OK"
2026-11-26 20:37:13,541 - INFO - HTTP Request: GET https://huggingface.co/api/models/cardiffnlp/twitter-xlm-roberta-base-sentiment/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-11-26 20:37:13,634 - INFO - HTTP Request: GET https://huggingface.co/api/models/cardiffnlp/twitter-xlm-roberta-base-sentiment/commits/refs%2Fpr%2F15 "HTTP/1.1 200 OK"
2026-11-26 20:37:14,120 - INFO - HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter


## 3. Recuperación y preparación de noticias

Se recuperan las noticias almacenadas en PostgreSQL, utilizando la tabla `rss_news` como fuente de datos.

El análisis de sentimientos se aplica independientemente de la clasificación temática, permitiendo reutilizar sus resultados para diferentes temas de monitoreo.

Para cada noticia se construye un texto de entrada mediante la concatenación del título y la descripción.

Los valores nulos se reemplazan por cadenas vacías y se eliminan espacios innecesarios. No se aplican transformaciones como eliminación de palabras vacías o lematización, con el objetivo de preservar el contexto lingüístico utilizado por el modelo Transformer.


In [8]:

def get_news_for_sentiment(connection):
    """
    Recupera las noticias disponibles para el análisis
    de sentimientos.

    Parameters
    ----------
    connection : psycopg.Connection
        Conexión activa a PostgreSQL.

    Returns
    -------
    pandas.DataFrame
        Noticias con identificador, título y descripción.
    """
    query = """
        SELECT
            rss_news_id,
            title,
            description
        FROM rss_news
        ORDER BY rss_news_id;
    """

    with connection.cursor() as cursor:
        cursor.execute(query)
        records = cursor.fetchall()
        columns = [
            column.name
            for column in cursor.description
        ]

    return pd.DataFrame(records, columns=columns)


connection = get_database_connection()

try:
    sentiment_news = get_news_for_sentiment(connection)
finally:
    connection.close()

logger.info(
    "Noticias recuperadas: %s",
    len(sentiment_news),
)

display(sentiment_news.head())


2026-11-26 20:37:14,885 - INFO - Noticias recuperadas: 50


,rss_news_id,title,description
0,1,Energía Costa Serena: amplía la generación solar,La puesta en marcha de 24 MW adicionales fue c...
1,2,Energía Costa Serena: reduce interrupciones el...,La mejora del servicio benefició a 800 hogares...
2,3,Energía Costa Serena: recupera capacidad de un...,La reparación permitió recuperar 18 MW y refor...
3,4,Energía Costa Serena: sufre una falla en una s...,El corte afectó a 1200 usuarios y generó recla...
4,5,Energía Costa Serena: demora obras de transpor...,El retraso de tres meses agravó la incertidumb...


In [9]:

def build_sentiment_text(row):
    """
    Construye el texto de entrada para el análisis
    de sentimientos a partir del título y descripción.

    Parameters
    ----------
    row : pandas.Series
        Registro correspondiente a una noticia.

    Returns
    -------
    str
        Texto preparado para la inferencia.
    """
    title = (
        str(row["title"]).strip()
        if pd.notna(row["title"])
        else ""
    )

    description = (
        str(row["description"]).strip()
        if pd.notna(row["description"])
        else ""
    )

    return ". ".join(
        part for part in [title, description] if part
    )


sentiment_news["sentiment_text"] = sentiment_news.apply(
    build_sentiment_text,
    axis=1,
)

# Identificar registros sin contenido procesable.
empty_text_count = (
    sentiment_news["sentiment_text"]
    .eq("")
    .sum()
)

logger.info(
    "Noticias sin texto procesable: %s",
    empty_text_count,
)

display(
    sentiment_news[
        [
            "rss_news_id",
            "sentiment_text",
        ]
    ].head(10)
)


2026-11-26 20:37:15,194 - INFO - Noticias sin texto procesable: 0


,rss_news_id,sentiment_text
0,1,Energía Costa Serena: amplía la generación sol...
1,2,Energía Costa Serena: reduce interrupciones el...
2,3,Energía Costa Serena: recupera capacidad de un...
3,4,Energía Costa Serena: sufre una falla en una s...
4,5,Energía Costa Serena: demora obras de transpor...
5,6,Energía Costa Serena: registra pérdidas de com...
6,7,Energía Costa Serena: publica el programa de m...
7,8,Energía Costa Serena: informa el volumen de ge...
8,9,Energía Costa Serena: presenta una solicitud d...
9,10,Energía Costa Serena: anuncia una audiencia ta...



## 4. Inferencia de sentimientos

Se aplica el modelo XLM-RoBERTa sobre los títulos y descripciones de las noticias recuperadas.

La inferencia se realiza por lotes para optimizar el procesamiento. Para cada noticia se obtiene una etiqueta de sentimiento predominante y las puntuaciones correspondientes a las categorías positiva, negativa y neutral.

Los resultados se mantienen temporalmente en memoria para verificar su integridad antes de almacenarlos en PostgreSQL.


In [10]:

def classify_news_sentiment(news, classifier, config):
    """
    Clasifica el sentimiento general de las noticias.

    Parameters
    ----------
    news : pandas.DataFrame
        Noticias con la columna sentiment_text.
    classifier : transformers.Pipeline
        Modelo de clasificación de sentimientos.
    config : dict
        Configuración de inferencia.

    Returns
    -------
    pandas.DataFrame
        Noticias con etiquetas y puntuaciones de sentimiento.
    """
    results = news.copy()

    if results.empty:
        raise ValueError("No hay noticias para clasificar.")

    predictions = classifier(
        results["sentiment_text"].tolist(),
        truncation=True,
        max_length=config["max_length"],
        batch_size=config["batch_size"],
        top_k=None,
    )

    scores = [
        {
            item["label"].lower(): float(item["score"])
            for item in prediction
        }
        for prediction in predictions
    ]

    score_columns = [
        "negative",
        "neutral",
        "positive",
    ]

    score_results = pd.DataFrame(
        scores,
        index=results.index,
    )[score_columns]

    score_results = score_results.rename(
        columns={
            "negative": "negative_score",
            "neutral": "neutral_score",
            "positive": "positive_score",
        }
    )

    results = pd.concat(
        [results, score_results],
        axis=1,
    )

    results["sentiment_label"] = (
        score_results.idxmax(axis=1)
        .str.replace("_score", "", regex=False)
    )

    results["sentiment_score"] = score_results.max(axis=1)

    return results


# Ejecutar inferencia sobre todas las noticias.
sentiment_results = classify_news_sentiment(
    news=sentiment_news,
    classifier=sentiment_classifier,
    config=SENTIMENT_CONFIG,
)

logger.info(
    "Noticias clasificadas: %s",
    len(sentiment_results),
)

display(
    sentiment_results[
        [
            "rss_news_id",
            "sentiment_label",
            "negative_score",
            "neutral_score",
            "positive_score",
        ]
    ].head(10).round(4)
)


2026-11-26 20:37:22,978 - INFO - Noticias clasificadas: 50


,rss_news_id,sentiment_label,negative_score,neutral_score,positive_score
0,1,positive,0.0170,0.1750,0.8081
1,2,positive,0.0227,0.1846,0.7927
2,3,positive,0.0162,0.2236,0.7603
3,4,negative,0.8144,0.1657,0.0199
4,5,negative,0.8482,0.1328,0.0190
5,6,negative,0.8138,0.1669,0.0193
6,7,neutral,0.0999,0.8385,0.0616
7,8,neutral,0.0325,0.8114,0.1562
8,9,neutral,0.0274,0.8487,0.1240
9,10,neutral,0.0166,0.8708,0.1127



## 5. Validación técnica de resultados

Se verifica la integridad de los resultados del análisis de sentimientos mediante controles de completitud, unicidad, etiquetas válidas y consistencia de las puntuaciones.

Estas verificaciones garantizan que los registros cumplen las condiciones necesarias para su almacenamiento, pero no constituyen una evaluación de la calidad predictiva del modelo.


In [11]:




def validate_sentiment_results(results, expected_count):
    """
    Verifica la integridad de los resultados de sentimiento.

    Parameters
    ----------
    results : pandas.DataFrame
        Resultados de la clasificación.
    expected_count : int
        Cantidad de noticias procesadas.

    Raises
    ------
    ValueError
        Si alguna validación de integridad falla.
    """
    score_columns = [
        "negative_score",
        "neutral_score",
        "positive_score",
    ]

    required_columns = [
        "rss_news_id",
        "sentiment_label",
        *score_columns,
    ]

    if len(results) != expected_count:
        raise ValueError("La cantidad de resultados no coincide.")

    if results[required_columns].isna().any().any():
        raise ValueError("Existen resultados con valores nulos.")

    if results["rss_news_id"].duplicated().any():
        raise ValueError("Existen noticias duplicadas.")

    if not results["sentiment_label"].isin(
        SENTIMENT_LABELS
    ).all():
        raise ValueError("Se encontraron etiquetas inválidas.")

    scores = results[score_columns].to_numpy(dtype=float)

    if not np.isfinite(scores).all():
        raise ValueError("Existen puntuaciones no finitas.")

    if not ((scores >= 0) & (scores <= 1)).all():
        raise ValueError("Existen puntuaciones fuera del rango [0, 1].")

    if not np.allclose(
        scores.sum(axis=1),
        1.0,
        atol=1e-4,
    ):
        raise ValueError("Las puntuaciones no suman aproximadamente 1.")

    logger.info("Todas las validaciones fueron satisfactorias.")


validate_sentiment_results(
    results=sentiment_results,
    expected_count=len(sentiment_news),
)


2026-11-26 20:37:42,638 - INFO - Todas las validaciones fueron satisfactorias.



## 6. Persistencia de resultados

Los resultados del análisis de sentimientos se almacenan en la tabla `rss_sentiments`, relacionada con `rss_news` mediante una clave foránea.

Se utiliza una operación UPSERT para insertar nuevas clasificaciones o actualizar las existentes, evitando duplicados.

La tabla conserva una clasificación por noticia, incluyendo la etiqueta, las puntuaciones del modelo, su identificador y la fecha de procesamiento.


In [12]:

UPSERT_SENTIMENT = """
INSERT INTO rss_sentiments (
    fk_rss_news_id,
    sentiment,
    positive_score,
    neutral_score,
    negative_score,
    model_name,
    processed_at
)
VALUES (
    %(rss_news_id)s,
    %(sentiment_label)s,
    %(positive_score)s,
    %(neutral_score)s,
    %(negative_score)s,
    %(model_name)s,
    NOW()
)
ON CONFLICT (fk_rss_news_id)
DO UPDATE SET
    sentiment = EXCLUDED.sentiment,
    positive_score = EXCLUDED.positive_score,
    neutral_score = EXCLUDED.neutral_score,
    negative_score = EXCLUDED.negative_score,
    model_name = EXCLUDED.model_name,
    processed_at = NOW();
"""


def save_sentiment_results(connection, results, model_name):
    """
    Inserta o actualiza los sentimientos de las noticias.

    Parameters
    ----------
    connection : psycopg.Connection
        Conexión activa a PostgreSQL.
    results : pandas.DataFrame
        Resultados validados de la clasificación.
    model_name : str
        Identificador del modelo utilizado.
    """
    records = results[
        [
            "rss_news_id",
            "sentiment_label",
            "positive_score",
            "neutral_score",
            "negative_score",
        ]
    ].to_dict(orient="records")

    for record in records:
        record["rss_news_id"] = int(record["rss_news_id"])
        record["model_name"] = model_name

        for column in [
            "positive_score",
            "neutral_score",
            "negative_score",
        ]:
            record[column] = float(record[column])

    with connection.cursor() as cursor:
        cursor.executemany(
            UPSERT_SENTIMENT,
            records,
        )

    logger.info(
        "Noticias preparadas para persistencia: %s",
        len(records),
    )


connection = get_database_connection()

try:
    save_sentiment_results(
        connection=connection,
        results=sentiment_results,
        model_name=SENTIMENT_CONFIG["model_name"],
    )

    connection.commit()

    logger.info(
        "Resultados almacenados correctamente."
    )

except Exception:
    connection.rollback()
    logger.exception(
        "Error al almacenar los sentimientos."
    )
    raise

finally:
    connection.close()


2026-11-26 20:37:44,752 - INFO - Noticias preparadas para persistencia: 50
2026-11-26 20:37:44,754 - INFO - Resultados almacenados correctamente.


In [13]:

connection = get_database_connection()

try:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT
                sentiment,
                COUNT(*) AS news_count
            FROM rss_sentiments
            WHERE model_name = %s
            GROUP BY sentiment
            ORDER BY sentiment;
            """,
            (SENTIMENT_CONFIG["model_name"],),
        )

        distribution = cursor.fetchall()

    display(
        pd.DataFrame(
            distribution,
            columns=["sentiment", "news_count"],
        )
    )

finally:
    connection.close()


,sentiment,news_count
0,negative,16
1,neutral,18
2,positive,16
